In [69]:
# Import libraries
import spacy
from spacy import displacy
import pandas as pd
nlp = spacy.load("en_core_web_sm")

In [83]:
# Define active and passive sentences.
active = ['Hens lay eggs.',
         'Birds build nests.',
         'The batter hit the ball.',
         'The computer transmitted a copy of the manual']

passive = ['Eggs are laid by hens',
           'Nests are built by birds',
           'The ball was hit by the batter',
           'A copy of the manual was transmitted by the computer.',
           'The cop and the thief story was being followed with keen interest by a curious public.']

### How do we impliment the rule `if dep nsubjpass, then passive else not`?

In [71]:
# Import matcher 
from spacy.matcher import Matcher

### Read more about it [here](https://spacy.io/api/matcher)

In [84]:
# Visualise the dependency parse tree of 1st sentence of the passive sentences.
docs_passive = [nlp(sent) for sent in passive]
option = {'color':'white', 'bg':'black'}
displacy.render(docs_passive, style='dep', options=option)

### Create a rule with `Matcher`

In [ ]:
#Create rule with matcher
rule = [{'POS':'NOUN'}]     # rule for matching nouns
matcher = Matcher(vocab= nlp.vocab)
matcher.add('Rule', [rule])

In [74]:
# demo
[(doc,  matcher(doc)) for doc in docs_passive]

[(Eggs are laid by hens,
  [(15740618714089435985, 0, 1), (15740618714089435985, 4, 5)]),
 (Nests are built by birds,
  [(15740618714089435985, 0, 1), (15740618714089435985, 4, 5)]),
 (The ball was hit by the batter,
  [(15740618714089435985, 1, 2), (15740618714089435985, 6, 7)]),
 (A copy of the manual was transmitted by the computer.,
  [(15740618714089435985, 1, 2),
   (15740618714089435985, 4, 5),
   (15740618714089435985, 9, 10)]),
 (The cop and the thief story was being followed with keen interest by a curious public.,
  [(15740618714089435985, 1, 2),
   (15740618714089435985, 4, 5),
   (15740618714089435985, 5, 6),
   (15740618714089435985, 11, 12),
   (15740618714089435985, 15, 16)])]

In [ ]:
# Sentences and the nouns
[{doc: [doc[match[-2]:match[-1]] for match in matcher(doc)]} for doc in docs_passive]

[{Eggs are laid by hens: [Eggs, hens]},
 {Nests are built by birds: [Nests, birds]},
 {The ball was hit by the batter: [ball, batter]},
 {A copy of the manual was transmitted by the computer.: [copy,
   manual,
   computer]},
 {The cop and the thief story was being followed with keen interest by a curious public.: [cop,
   thief,
   story,
   interest,
   public]}]

In [ ]:
# Active sentences and nouns
docs_active = [nlp(sent) for sent in active]

[{doc: [doc[match[-2]:match[-1]] for match in matcher(doc)]} for doc in docs_active]

[{Hens lay eggs.: [Hens, eggs]},
 {Birds build nests.: [Birds, nests]},
 {The batter hit the ball.: [batter, ball]},
 {The computer transmitted a copy of the manual: [computer, copy, manual]}]

### Create a rule for `passive voice`

In [79]:
# Rule for passive subjects
rule = [{'DEP':'nsubjpass'}]
matcher = Matcher(vocab= nlp.vocab)
matcher.add('Rule', [rule])

In [102]:
[{doc: [doc[match[-2]:match[-1]] for match in matcher(doc)]} for doc in docs_passive]

[{Eggs are laid by hens: [Eggs]},
 {Nests are built by birds: [Nests]},
 {The ball was hit by the batter: [ball]},
 {A copy of the manual was transmitted by the computer.: [copy]},
 {The cop and the thief story was being followed with keen interest by a curious public.: [cop,
   story]}]

### Let's check how this rule works if we use it on a sentence with `active voice`

In [103]:
[{doc: matcher(doc)} for doc in docs_active]

[{Hens lay eggs.: []},
 {Birds build nests.: []},
 {The batter hit the ball.: []},
 {The computer transmitted a copy of the manual: []}]

### Now lets make a function that impliments this logic

In [119]:
def is_passive(doc, matcher):

    if len(matcher(doc))>0:
        return True
    else:
        return False

### Let's test this function on our small sample of sentences and see how the pipeline will work

In [120]:
rule_passive = [{'DEP': 'nsubjpass'}]
passive = Matcher(vocab= nlp.vocab)
passive.add('Rule', [rule_passive])

[{doc: is_passive(doc, passive)} for doc in docs_passive]

[{Eggs are laid by hens: True},
 {Nests are built by birds: True},
 {The ball was hit by the batter: True},
 {A copy of the manual was transmitted by the computer.: True},
 {The cop and the thief story was being followed with keen interest by a curious public.: True}]

In [121]:
[{doc: is_passive(doc, passive)} for doc in docs_active]

[{Hens lay eggs.: False},
 {Birds build nests.: False},
 {The batter hit the ball.: False},
 {The computer transmitted a copy of the manual: False}]

### Summary
 - One can go a long way by observing patterns in linguistic data, you don't always need to know the details of the linguitsics very well.
 - Once can use the `matcher` object to find if certain linguistic patterns exist in data

Practice exercise

In [123]:
import spacy
from spacy.matcher import Matcher
nlp = spacy.load("en_core_web_sm")

rule = [{'DEP':'nsubjpass'}]
matcher = Matcher(nlp.vocab)
matcher.add('Rule',[rule])

doc = nlp('A book is being bought by John.')
matcher(doc)

[(15740618714089435985, 1, 2)]